# Лабораторная работа №2
## Обнаружение аномалий и выбросов в данных

**Датасет:** `Housing.csv`

Цель: выполнить профилирование числового датасета, визуально исследовать выбросы, проверить нормальность распределений, применить Z-score и IQR, а затем выполнить многомерное обнаружение аномалий по расстояниям до ближайших соседей.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = pd.read_csv("Housing.csv")
display(df.head())

## Задание 1. Загрузка и профилирование данных

In [ ]:
print("Размер датасета:", df.shape)
df.info()

print("\nОписательная статистика:")
display(df.describe(include="all").T)

print("\nПропуски:")
display(df.isna().sum().to_frame("missing"))

print("\nДубликаты:", df.duplicated().sum())

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("\nЧисловые признаки:", numeric_cols)

display(pd.DataFrame({
    "min": df[numeric_cols].min(),
    "max": df[numeric_cols].max(),
    "range": df[numeric_cols].max() - df[numeric_cols].min()
}))

### Визуальный анализ

Для детального анализа выбраны признаки **`price`** и **`area`**. Для каждого строятся гистограмма и boxplot.

In [ ]:
for col in ["price", "area"]:
    fig, ax = plt.subplots()
    ax.hist(df[col].dropna(), bins=30, edgecolor="black")
    ax.set_title(f"Гистограмма: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Частота")
    plt.show()

    fig, ax = plt.subplots()
    ax.boxplot(df[col].dropna(), vert=False)
    ax.set_title(f"Boxplot: {col}")
    ax.set_xlabel(col)
    plt.show()

### Вывод

Гистограммы показывают форму распределений, а boxplot позволяет увидеть медиану, квартильный диапазон и потенциальные экстремальные наблюдения. Точки за пределами «усов» рассматриваются как кандидаты на выбросы и далее проверяются статистическими методами.

## Задание 2. Проверка нормальности распределения

In [ ]:
selected_features = ["price", "area"] if "price" in df.columns and "area" in df.columns else numeric_cols[:2]

rows = []
for col in selected_features:
    x = df[col].dropna()
    sh = stats.shapiro(x)
    rows.append({
        "feature": col,
        "mean": x.mean(),
        "median": x.median(),
        "std": x.std(),
        "skewness": x.skew(),
        "kurtosis": x.kurtosis(),
        "shapiro_stat": sh.statistic,
        "p_value": sh.pvalue
    })

normality = pd.DataFrame(rows)
display(normality)

In [ ]:
for col in selected_features:
    x = df[col].dropna()
    fig = plt.figure()
    ax = fig.add_subplot(111)
    stats.probplot(x, dist="norm", plot=ax)
    ax.set_title(f"QQ-plot: {col}")
    plt.show()

### Критерий Шапиро–Уилка

**H0:** выборка подчиняется нормальному распределению.

**H1:** выборка не подчиняется нормальному распределению.

Уровень значимости: **α = 0.05**.

Если `p-value < 0.05`, H0 отвергается; если `p-value >= 0.05`, оснований отвергать H0 нет.

In [ ]:
alpha = 0.05
normality["decision"] = np.where(
    normality["p_value"] < alpha,
    "Отклоняем H0",
    "Нет оснований отвергать H0"
)
display(normality[["feature", "shapiro_stat", "p_value", "decision"]])

### Вывод

Решение о нормальности принимается отдельно для каждого признака по p-value критерия Шапиро–Уилка и по характеру отклонений на QQ-графике.

## Задание 3. Обнаружение выбросов статистическими методами

### 3.1. Z-score для `price`

Критерий выброса: **|Z| > 3**.

In [ ]:
x = df["price"].dropna()
z = pd.Series(stats.zscore(x), index=x.index)
z_outliers = df.loc[z.abs() > 3].copy()

print("Количество выбросов:", len(z_outliers))
display(z_outliers)

fig, ax = plt.subplots()
ax.scatter(z.index, z.values, s=20)
ax.axhline(3, linestyle="--")
ax.axhline(-3, linestyle="--")
ax.set_title("Z-score: price")
ax.set_xlabel("Индекс")
ax.set_ylabel("Z-score")
plt.show()

### 3.2. Метод IQR для `price`

In [ ]:
q1 = df["price"].quantile(0.25)
q3 = df["price"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

iqr_mask = (df["price"] < lower) | (df["price"] > upper)
iqr_outliers = df.loc[iqr_mask].copy()

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Нижняя граница:", lower)
print("Верхняя граница:", upper)
print("Количество выбросов:", len(iqr_outliers))
display(iqr_outliers)

In [ ]:
comparison = pd.DataFrame({
    "method": ["Z-score", "IQR"],
    "outlier_count": [len(z_outliers), len(iqr_outliers)]
})
display(comparison)

fig, ax = plt.subplots()
ax.bar(comparison["method"], comparison["outlier_count"])
ax.set_title("Сравнение количества выбросов")
ax.set_ylabel("Количество")
plt.show()

### Вывод

Z-score определяет экстремальные значения относительно среднего и стандартного отклонения. IQR использует квартильный диапазон и менее зависит от экстремальных наблюдений. Количество найденных выбросов может различаться.

## Задание 4. Метод локальной плотности / ближайших соседей

Для многомерного анализа используются все числовые признаки. Перед расчётом расстояний выполняется `StandardScaler`.

Для `k = 5, 7, 9, 10` вычисляется среднее расстояние до k ближайших соседей. Верхние 5% значений считаются аномалиями.

In [ ]:
multi_features = df.select_dtypes(include=np.number).columns.tolist()
X = df[multi_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Признаки:", multi_features)
print("Размер матрицы:", X_scaled.shape)

### Евклидово расстояние

In [ ]:
euclidean_results = []
euclidean_metrics = {}

for k in [5, 7, 9, 10]:
    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
    nn.fit(X_scaled)
    distances, _ = nn.kneighbors(X_scaled)
    mean_dist = distances[:, 1:k+1].mean(axis=1)
    threshold = np.quantile(mean_dist, 0.95)
    anomaly = mean_dist >= threshold

    euclidean_metrics[k] = mean_dist
    euclidean_results.append({
        "k": k,
        "threshold_95%": threshold,
        "anomaly_count": int(anomaly.sum()),
        "anomaly_share_%": anomaly.mean() * 100
    })

euclidean_summary = pd.DataFrame(euclidean_results)
display(euclidean_summary)

In [ ]:
for k, values in euclidean_metrics.items():
    fig, ax = plt.subplots()
    ax.hist(values, bins=30, edgecolor="black")
    ax.axvline(np.quantile(values, 0.95), linestyle="--")
    ax.set_title(f"Среднее евклидово расстояние, k={k}")
    ax.set_xlabel("Среднее расстояние")
    ax.set_ylabel("Частота")
    plt.show()

### Манхэттенское расстояние

In [ ]:
manhattan_results = []
manhattan_metrics = {}

for k in [5, 7, 9, 10]:
    nn = NearestNeighbors(n_neighbors=k+1, metric="manhattan")
    nn.fit(X_scaled)
    distances, _ = nn.kneighbors(X_scaled)
    mean_dist = distances[:, 1:k+1].mean(axis=1)
    threshold = np.quantile(mean_dist, 0.95)
    anomaly = mean_dist >= threshold

    manhattan_metrics[k] = mean_dist
    manhattan_results.append({
        "k": k,
        "threshold_95%": threshold,
        "anomaly_count": int(anomaly.sum()),
        "anomaly_share_%": anomaly.mean() * 100
    })

manhattan_summary = pd.DataFrame(manhattan_results)
display(manhattan_summary)

In [ ]:
for k, values in manhattan_metrics.items():
    fig, ax = plt.subplots()
    ax.hist(values, bins=30, edgecolor="black")
    ax.axvline(np.quantile(values, 0.95), linestyle="--")
    ax.set_title(f"Среднее манхэттенское расстояние, k={k}")
    ax.set_xlabel("Среднее расстояние")
    ax.set_ylabel("Частота")
    plt.show()

### Scatter plot аномалий

Для визуализации используются `area` и `price`.

In [ ]:
visual_k = 7
d = euclidean_metrics[visual_k]
threshold = np.quantile(d, 0.95)
mask = d >= threshold

fig, ax = plt.subplots()
ax.scatter(df["area"], df["price"], s=25, alpha=0.6)
ax.scatter(
    df.loc[mask, "area"],
    df.loc[mask, "price"],
    s=45,
    label="Аномалии — верхние 5%"
)
ax.set_xlabel("area")
ax.set_ylabel("price")
ax.set_title(f"Аномалии по евклидову расстоянию, k={visual_k}")
ax.legend()
plt.show()

### Анализ устойчивости при изменении k

In [ ]:
stability = euclidean_summary.merge(
    manhattan_summary,
    on="k",
    suffixes=("_euclidean", "_manhattan")
)
display(stability)

## Итоговые выводы

В работе выполнено профилирование `Housing.csv`, визуальный анализ двух числовых признаков, проверка нормальности критерием Шапиро–Уилка, обнаружение выбросов методами Z-score и IQR, а также многомерный поиск аномалий на основе расстояний до ближайших соседей.

Для метрического анализа признаки стандартизированы. Рассмотрены евклидово и манхэттенское расстояния и значения `k = 5, 7, 9, 10`. Верхние 5% объектов по средней дистанции интерпретируются как потенциальные аномалии.

Статистически необычное значение не обязательно является ошибкой, поэтому решение об удалении или изменении наблюдений должно учитывать предметную область.